In [40]:
!pip -q install transformers pandas

GPU and Drive : 1

In [41]:
import os, json, numpy as np, pandas as pd, torch
assert torch.cuda.is_available(), "GPU missing: Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))
from google.colab import drive; drive.mount("/content/drive")
D = "/content/drive/MyDrive/spoilfree/"
print(os.listdir(D))

Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['val.csv', 'test._heldout.csv', 'val_scored.csv', 'train_scored.csv', 'test_scored.csv', 'clean_train.csv', 'clean_val.csv', 'clean_test.csv', 'IMDB_reviews.json', 'IMDB_movie_details.json']


Settings : 2

In [42]:
teacher="bhavyagiri/roberta-base-finetuned-imdb-spoilers"
student="roberta-base"
maxlen=160
keep_pos=0.30 #keep top 30% according to teacher
keep_neg=0.90 #keep 70% of the least spoiler content
epochs=3
bs=32
lr=2e-5
seed=1

try:
    test_file=D+"test_heldout.csv"
    if not os.path.exists(test_file):
      test_file=D+'test._heldout.csv'
    if not os.path.exists(test_file):
        raise FileNotFoundError("test._heldout.csv not found")
    print("Using:", test_file)
except FileNotFoundError as e:
    print(f"File not found: {e}")

torch.manual_seed(seed)
np.random.seed(seed)

Using: /content/drive/MyDrive/spoilfree/test._heldout.csv


**Helper Functions** : 3

In [43]:
from sys import thread_info
from sklearn.metrics import roc_auc_score, average_precision_score

def clean_split(d,keep_posnew=None,keep_negnew=None,seed=1):
  keep_posnew=keep_pos if keep_posnew is None else keep_posnew
  keep_negnew=keep_neg if keep_negnew is None else keep_negnew
  pos,neg=d[d.label==1],d[d.label==0]
  pos=pos[pos.t1>=pos.t1.quantile(1-keep_posnew)]
  neg=neg[neg.t1>=neg.t1.quantile(keep_negnew)]
  k=min(len(pos),len(neg))
  out=pd.concat(pos.sample(k,random_state=seed),neg.sample(k,random_state=seed))
  return out.sample(frac=1,random_state=seed).reset_index(drop=True)[["text","label","t1"]]


def sweep(y,p,name,ths=(0.3,0.5,0.7,0.8,0.9,0.95)):
  y=np.asarray(y)
  p=np.asarray(p)
  print(f"{name} AUC: {roc_auc_score(y,p):.3f} AP {average_precision_score(y,p):.3f}")
  print(f"thr precision recall falseblur missed")
  for ts in ths:
    pr=p>=ts
    tp=int((pr & (y==1)).sum())
    fp=int((pr & (y==0)).sum())       #confusion matrix metric
    fn=int(((~pr) & (y==1)).sum())
    tn=int(((~pr) & (y==0)).sum())
    print(f"{ts:<6} {tp / max(tp + fp, 1):<10.3f} {tp / max(tp + fn, 1):<7.3f} {fp:<10} {fn}")


def pick_threshold(y,p,min_precision=0.95):
  #lowest precision that is >=min_precision if no threshold reaches then None
  y=np.asarray(y)
  p=np.asarray(p)
  for ts in np.arange(0.05,1.0,0.01):
    pr=p>=ts
    if pr.sum() and (pr&(y==1)).sum()/pr.sum()>=min_precision:
      return round(float(ts),2)
  return None




**Score from the teacher** : 4

In [44]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
ttok=AutoTokenizer.from_pretrained(teacher)    #loads tokenizer to convert text into model readable tokens
teacher=AutoModelForSequenceClassification.from_pretrained(teacher).half().cuda().eval()  #loads pretrained teacher model
                                                                                            #then uses half precision to reduce memory usage,
                                                                                            #moves model to gpu and then enables evaluation mode
                                                                                            #disabling training spiecific behaviour like a dropout

@torch.no_grad()  #disables gradient calculations: saves memory and computation
def score_with(model,tokenizer,texts,bs=128):
  #processes upto 128 texts per batch


  lengths=[len(t) for t in texts]            #calculates the length of each text
  order=np.argsort(lengths)                    #sorts indices by text length in ascending order
  out=np.zeros(len(texts),dtype=np.float32)             #creates an array to store spoiler scores

  for i in range(0,len(texts),bs):                  #iterates through texts in batches of size bs
    j=order[i:i+bs]                                #gets sorted indices for the current batch
    batch_texts=[texts[k] for k in j]                     #retrieves texts using the sorted indices

    x=tokenizer(                             #converts batch texts into model-readable tokens
        batch_texts,                         #passes the current batch of texts
        truncation=True,                         #truncates texts exceeding the token limit
        max_length=maxlen,                     #sets the maximum number of tokens per text
        padding=True,                         #pads shorter sequences to match the longest sequence in the batch
        return_tensors="pt"                     #returns tokenized data as pytorch tensors
    )

    x=x.to("cuda")       #moves tokenized tensors to the gpu
    logits=model(**x).logits      #gets raw prediction scores from the model
    logits=logits.float()          #converts logits to 32-bit floating-point values

    probabilities=torch.softmax(logits,dim=1)          #converts logits into class probabilities
    spoiler_scores=probabilities[:,1].cpu().numpy()       #extracts class 1 probabilities and converts them to a numpy array
    out[j]=spoiler_scores                                 #stores scores at their original text indices

  return out


for n,f in [("val","val.csv"),("test",test_file),("train","train.csv")]:     #iterates through validation, test, and training datasets
 out=D+f"{n}_scored.csv"       #creates the output file path
 if os.path.exists(out):
   print(f"{n} already scored")
   continue                                                            #skips to the next dataset

 path=f if os.path.isabs(f) else os.path.join(D,f)  #uses the full path if f is absolute

 if not os.path.exists(path):
   print(f"{n} file not found: {path}")  #prints the missing file path
   print("Available files:",os.listdir(D))  #lists files in the dataset directory
   continue                       #skips to the next dataset

 d=pd.read_csv(path)  #reads the dataset from the correct path
 d["t1"]=score_with(teacher,ttok,d.text.tolist())    #calculates teacher spoiler probabilities for each text
 d.to_csv(out,index=False)
 print(f"{n} scored {len(d)}")

del teacher        #deletes the teacher model variable
torch.cuda.empty_cache()


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

val already scored
test already scored
train already scored


**create new clean dataset** : 5a


In [45]:
def clean_dataset(df,extra=None):
 spoilers=df[df.label==1]
 non_spoilers=df[df.label==0]

 spoiler_cutoff=keep_pos #selecting those rows only where teacher thinks has the highest spoiler
 spoilers=spoilers[spoilers.t1>=spoiler_cutoff]
 non_spoiler_cutoff=non_spoilers.t1.quantile(keep_neg)
 non_spoilers=non_spoilers[non_spoilers.t1<=non_spoiler_cutoff]   #rows where the teacher thinks has the lowest spoielrs

 if extra is not None:   #extra rows from the movie file (Cell 5c)
  spoilers=pd.concat([spoilers,extra[extra.label==1]])
  non_spoilers=pd.concat([non_spoilers,extra[extra.label==0]])

 n=min(len(spoilers),len(non_spoilers))     #equal rows in both classes
 spoilers=spoilers.sample(n,random_state=seed)
 non_spoilers=non_spoilers.sample(n,random_state=seed)


 both=pd.concat([spoilers,non_spoilers])
 both=both.sample(frac=1,random_state=seed)    #shuffles them

 return both[["text","label","t1"]].reset_index(drop=True)


clean={}
for name in ["train","val","test"]:
 scored=pd.read_csv(D+name+"_scored.csv")
 clean[name]=clean_dataset(scored)
 clean[name].to_csv(D+"clean_"+name+".csv",index=False)
 print(f"{name} rows {len(clean[name])} | per class: {len(clean[name])//2}")

train rows 6140 | per class: 3070
val rows 648 | per class: 324
test rows 770 | per class: 385


In [52]:
#5b

train_rows = clean["train"]     #checking the top 12 spoilers and non spoiler found

for label in [1, 0]:
    print("\nlabel", label, "(1 = spoiler, 0 = safe\n")
    sample = train_rows[train_rows.label == label].sample(12, random_state=3)
    for text, score in zip(sample.text, sample.t1):
        print(round(score, 3), "|", text[:170])


label 1 (1 = spoiler, 0 = safe

0.315 | Bane orders his men to round up Fox and the others for judgment and to hang the soldiers' bodies from one of the bridges as a message for the world.
0.429 | He takes Louise to his car and has her sit in the front seat. As he picks up his radio, Thelma intervenes, with a gun.
0.897 | The twist in the end with the doctor being a serial killer was unnecessary and brought nothing to the overall movie. Also there was absolutely no character development bu
0.723 | When Razieh confessed to her that the miscarriage was caused by the car accident and not by falling down the stairs, Simin still made him pay the money because she knew i
0.503 | The suicide attempt for me was the peak of the story! The chemistry between Brie and Jacob on and off set is undeniable.
0.578 | A bloody shootout between them and Dog's gang leaves everybody dead. The money is given to Harry by his henchman Big Chris (Vinnie Jones) and his son Little Chris (Peter 
0.909 | He died, 

In [50]:
#5c

import json
import hashlib
import re
from transformers import AutoTokenizer,AutoModelForSequenceClassification

teacher_id="bhavyagiri/roberta-base-finetuned-imdb-spoilers"     #model name only, "teacher" is the model variable in cell 4
anchor_max=3000     #extra rows per class, at most
anchor_pool=8000     #candidates per class, the teacher keeps the ones it agrees with


def split_of(movie_id):
 h=int(hashlib.md5(movie_id.encode()).hexdigest(),16)%100     #same split as the dataset so val and test movies stay untouched
 if h<80:
  return "train"
 if h<90:
  return "val"
 return "test"


def split_sentences(text):
 text=re.sub(r"\s+"," ",text or "")
 text=re.sub(r"([a-z0-9\)\"'])([.!?])(?=[A-Z])",r"\1\2 ",text)     #glued sentences like end.Next
 return [s for s in re.split(r"(?<=[.!?])\s+",text) if s]


def is_junk(s):
 s=s.lower()
 for w in ["written by","@","<",">","http","imdb"]:     #credit lines at the end of synopses
  if w in s:
   return True
 return False



spoiler_rows=[]
safe_rows=[]
for line in open(D+"IMDB_movie_details.json",encoding="utf-8"):
 movie=json.loads(line)
 if split_of(movie["movie_id"])!="train":
  continue



 synopsis=split_sentences(movie.get("plot_synopsis"))
 for i,s in enumerate(synopsis):
  if not 40<=len(s)<=500 or is_junk(s):
   continue
  if i>=0.7*len(synopsis):
   spoiler_rows.append((s,1))     #last 30% of the synopsis, what happens and how it ends
  elif i<0.25*len(synopsis):
   safe_rows.append((s,0))     #first 25%, setup



 for s in split_sentences(movie.get("plot_summary")):
  if 40<=len(s)<=500 and not is_junk(s):
   safe_rows.append((s,0))     #short summary, premise only


print("found",len(spoiler_rows),"spoiler rows and",len(safe_rows),"safe rows")



pool=pd.concat([
 pd.DataFrame(spoiler_rows,columns=["text","label"]).sample(anchor_pool,random_state=seed),
 pd.DataFrame(safe_rows,columns=["text","label"]).sample(anchor_pool,random_state=seed)]).reset_index(drop=True)



teacher_tok=AutoTokenizer.from_pretrained(teacher_id)
teacher_model=AutoModelForSequenceClassification.from_pretrained(teacher_id,use_safetensors=True).half().cuda().eval()


texts=pool.text.tolist()
scores=[]
for start in range(0,len(texts),128):
 enc=teacher_tok(texts[start:start+128],truncation=True,max_length=maxlen,padding=True,return_tensors="pt").to("cuda")
 with torch.no_grad():
  logits=teacher_model(**enc).logits
 scores+=torch.softmax(logits.float(),dim=1)[:,1].cpu().tolist()
pool["t1"]=scores



del teacher_model
torch.cuda.empty_cache()

#same rule as the review rows, keep only what the teacher agrees with
spoiler_anchors=pool[(pool.label==1)&(pool.t1>=keep_pos)]
safe_anchors=pool[(pool.label==0)&(pool.t1<=pool[pool.label==0].t1.quantile(keep_neg))]
print("teacher kept",len(spoiler_anchors),"spoiler rows and",len(safe_anchors),"safe rows")

spoiler_anchors=spoiler_anchors.sample(min(anchor_max,len(spoiler_anchors)),random_state=seed)
safe_anchors=safe_anchors.sample(min(anchor_max,len(safe_anchors)),random_state=seed)
anchors=pd.concat([spoiler_anchors,safe_anchors])

train_scored=pd.read_csv(D+"train_scored.csv")
clean["train"]=clean_dataset(train_scored,extra=anchors)
clean["train"].to_csv(D+"clean_train.csv",index=False)
print(f"train rows {len(clean['train'])} | per class: {len(clean['train'])//2}")

found 26621 spoiler rows and 28796 safe rows


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

teacher kept 1942 spoiler rows and 7200 safe rows
train rows 10024 | per class: 5012


In [51]:
import os

print([f for f in os.listdir(D) if f.endswith(".json")])

['IMDB_reviews.json', 'IMDB_movie_details.json']


**Loading new bodel roberta based** : 6a

In [53]:
from transformers import AutoTokenizer,AutoModelForSequenceClassification,get_linear_schedule_with_warmup
from sklearn.metrics import roc_auc_score,average_precision_score

tok=AutoTokenizer.from_pretrained(student)
model=AutoModelForSequenceClassification.from_pretrained(student,num_labels=2,id2label={0:"not_spoiler",1:"spoiler"},label2id={"not_spoiler":0,"spoiler":1},use_safetensors=True).cuda()     #safetensors only, pickle files can run code

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [56]:
#6b helper functions

def make_batches(texts,labels=None,shuffle=False,batch_size=bs):
 if shuffle:
  order=np.random.permutation(len(texts))                #random order for training
 else:
  order=np.argsort([len(t) for t in texts])              #similar lengths together, faster

 for start in range(0,len(order),batch_size):            #iterates through texts in batches
  pos=order[start:start+batch_size]                      #gets indices for the current batch
  enc=tok([texts[i] for i in pos],truncation=True,max_length=maxlen,padding=True,return_tensors="pt").to("cuda")  #tokenizes texts and moves tensors to gpu
  y=None
  if labels is not None:
   y=torch.tensor([int(labels[i]) for i in pos]).cuda()  #labels for this batch
  yield pos,enc,y

@torch.no_grad()
def predict_probs(texts):
 model.eval()                                            #turns off dropout
 probs=np.zeros(len(texts),dtype=np.float32)
 for pos,enc,_ in make_batches(texts,batch_size=128):
  with torch.autocast("cuda",dtype=torch.float16):
   logits=model(**enc).logits
  probs[pos]=torch.softmax(logits.float(),dim=1)[:,1].cpu().numpy()  #spoiler probability per text
 return probs

In [57]:
#6c training

train_texts=clean["train"].text.tolist()
train_labels=clean["train"].label.tolist()
val_texts=clean["val"].text.tolist()
val_labels=clean["val"].label.values

optimizer=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=0.01)   #creates optimizer with weight decay
total_steps=epochs*((len(train_texts)+bs-1)//bs)                          #calculates total training steps
scheduler=get_linear_schedule_with_warmup(optimizer,int(0.06*total_steps),total_steps)  #warms up learning rate, then decreases it
scaler=torch.amp.GradScaler("cuda")                                       #fp16, faster on the t4


save_path=D+"model_v1"
best_score=-1                                               #tracks the best validation average precision

for epoch in range(1,epochs+1):
 model.train()                                                    #enables training mode
 loss_sum=0
 steps=0

 for pos,enc,y in make_batches(train_texts,train_labels,shuffle=True):  #processes shuffled training batches
  with torch.autocast("cuda",dtype=torch.float16):                           #uses mixed precision to speed up training
   logits=model(**enc).logits                                             #gets raw prediction scores
  loss=torch.nn.functional.cross_entropy(logits.float(),y)                 #calculates classification loss



  optimizer.zero_grad()                                                      #clears gradients from the previous step
  scaler.scale(loss).backward()                                                  #calculates gradients using scaled loss
  scaler.unscale_(optimizer)                                                     #unscales gradients before clipping
  torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)                         #clips gradients to prevent excessively large updates
  scaler.step(optimizer)                                                         #updates model parameters
  scaler.update()                                                             #updates the gradient scaler
  scheduler.step()                                                          #updates the learning rate

  loss_sum+=loss.item()
  steps+=1
  if steps%200==0:
   print("epoch",epoch,"step",steps,"loss",round(loss_sum/steps,4))  #prints average training loss



 val_probs=predict_probs(val_texts)                                           #predicts spoiler probabilities on validation data
 auc=roc_auc_score(val_labels,val_probs)                                          #calculates validation ROC-AUC
 ap=average_precision_score(val_labels,val_probs)                              #calculates validation average precision
 print("epoch",epoch,"done | val auc",round(auc,3),"| val ap",round(ap,3))

 if ap>best_score:                                                                  #keep only the best epoch
  best_score=ap
  model.save_pretrained(save_path)                                                   #saves the best model weights and configuration
  tok.save_pretrained(save_path)                                                  #saves the tokenizer
  print("saved to",save_path)

model=AutoModelForSequenceClassification.from_pretrained(save_path,use_safetensors=True).cuda()     #load the best epoch back

/tmp/ipykernel_921/1936913785.py:35: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()                                                          #updates the learning rate


epoch 1 step 200 loss 0.0787
epoch 1 done | val auc 0.998 | val ap 0.998


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved to /content/drive/MyDrive/spoilfree/model_v1
epoch 2 step 200 loss 0.0341
epoch 2 done | val auc 0.999 | val ap 0.999


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved to /content/drive/MyDrive/spoilfree/model_v1
epoch 3 step 200 loss 0.0165
epoch 3 done | val auc 1.0 | val ap 1.0


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved to /content/drive/MyDrive/spoilfree/model_v1


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

**2 functions for final search : 7a**


In [58]:
def print_table(y_true,probs,title):
 y_true=np.array(y_true)                                                         #converts true labels to a numpy array
 probs=np.array(probs)                                                             #converts predicted probabilities to a numpy array
 print("\n"+title,"| auc",round(roc_auc_score(y_true,probs),3))                     #prints title and ROC-AUC score
 print("threshold  precision  recall  false_blurs  missed")                       #prints column headings

 for t in [0.3,0.5,0.7,0.8,0.9,0.95]:
  flagged=probs>=t                                                                 #marks texts as spoilers if probability meets the threshold
  tp=(flagged&(y_true==1)).sum()                                                       #counts correctly identified spoilers
  fp=(flagged&(y_true==0)).sum()                                                    #safe text blurred by mistake
  fn=(~flagged&(y_true==1)).sum()                                                      #spoilers missed
  precision=tp/max(tp+fp,1)                                                        #fraction of flagged texts that are actual spoilers
  recall=tp/max(tp+fn,1)                                                           #fraction of actual spoilers correctly flagged
  print(f"{t:<10} {precision:<10.3f} {recall:<7.3f} {fp:<12} {fn}")                #prints metrics for the current threshold


def find_threshold(y_true,probs,min_precision=0.95):
 y_true=np.array(y_true)                                                          #converts true labels to a numpy array
 probs=np.array(probs)                                                            #converts predicted probabilities to a numpy array

 for t in np.arange(0.05,1.0,0.01):                                                     #tests thresholds from 0.05 to 0.99
  flagged=probs>=t                                                                 #marks texts whose probability meets the threshold
  if flagged.sum()==0:
   continue

  precision=(flagged&(y_true==1)).sum()/flagged.sum()                              #calculates precision among flagged texts
  if precision>=min_precision:
   return round(float(t),2)                                                        #returns the first threshold meeting the minimum precision

 return None  #returns None if no threshold meets the required precision

In [60]:
#7b final search

test_texts=clean["test"].text.tolist()
print_table(clean["test"].label,predict_probs(test_texts),"clean held-out movies")     #easy set, teacher picked these labels

hw=pd.read_csv(D+"eval_handwritten_107.csv")
hw["p"]=predict_probs(hw.text.tolist())
print_table(hw.label,hw.p,"hand-written 107")     #this is the real number

if "category" in hw.columns:
 hw["blurred"]=hw.p>=0.9
 print("\nshare blurred at 0.9 per category, safe should be near 0 and spoiler near 1")
 print(hw.groupby(["category","label"]).blurred.mean().round(2))


clean held-out movies | auc 1.0
threshold  precision  recall  false_blurs  missed
0.3        0.992      0.997   3            1
0.5        0.992      0.997   3            1
0.7        0.992      0.997   3            1
0.8        0.992      0.997   3            1
0.9        0.992      0.997   3            1
0.95       0.992      0.997   3            1

hand-written 107 | auc 0.984
threshold  precision  recall  false_blurs  missed
0.3        1.000      0.932   0            3
0.5        1.000      0.932   0            3
0.7        1.000      0.932   0            3
0.8        1.000      0.932   0            3
0.9        1.000      0.932   0            3
0.95       1.000      0.932   0            3

share blurred at 0.9 per category, safe should be near 0 and spoiler near 1
category                  label
edge_common_word          0        0.00
edge_non_english          0        0.00
edge_non_english_spoiler  1        0.00
edge_safe_tag             0        0.00
edge_spoiler_no_name      1 

In [61]:
#7c setting a threshold and taking numbers from content.js

import json

threshold=find_threshold(hw.label,hw.p)
print("threshold for 95% precision:",threshold)

if threshold is None:
 print("no threshold reaches 95% precision, do not ship this model")
else:
 bars={"beyond":max(0.5,round(threshold-0.10,2)),"unknown":threshold,"unknownPage":min(0.99,round(threshold+0.02,2))}
 print("put this in content.js: const BAR =",bars)
 with open(save_path+"/threshold.json","w") as f:
  json.dump({"threshold":threshold,"BAR":bars},f)

mistakes=hw[(hw.p>=0.9)&(hw.label==0)][["text","p"]]             #safe rows still blurred
mistakes.to_csv(D+"new_model_false_blurs.csv",index=False)
print("safe rows still blurred at 0.9:",len(mistakes))

threshold for 95% precision: 0.05
put this in content.js: const BAR = {'beyond': 0.5, 'unknown': 0.05, 'unknownPage': 0.07}
safe rows still blurred at 0.9: 0


**Download model zip**

In [63]:
import shutil
from google.colab import files

shutil.make_archive(D+"model_v1","zip",root_dir=save_path)
files.download(D+"model_v1.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>